In [1]:
import pandas as pd
df=pd.read_csv('/workspaces/BlizzardX/Data/cleaned_data.csv')

In [2]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [3]:
from src.Model.feature_engineering import FeatureEngineering , ColdEventDetector
fe=FeatureEngineering(df)
ce =ColdEventDetector(fe.apply_all_features())

In [4]:
df=ce.apply_cold_event_detection()

In [7]:
df.to_csv('/workspaces/BlizzardX/Data/feature_data.csv', index=False)

In [5]:
df[df['Cold_Event'] == 1]['Cold_Event'].count(),df[df['Cold_Event'] == 0]['Cold_Event'].count()

(np.int64(12334), np.int64(41517))

In [10]:
import pandas as pd
df=pd.read_csv('/workspaces/BlizzardX/Data/feature_data.csv')
df['DATE'] = pd.to_datetime(df['DATE'])

In [12]:
train_end_date = pd.to_datetime('2020-01-01')
val_end_date = pd.to_datetime('2021-01-01')

# Split into train, validation, and test
train_df = df[df['DATE'] < train_end_date]
val_df = df[(df['DATE'] >= train_end_date) & (df['DATE'] < val_end_date)]
test_df = df[df['DATE'] >= val_end_date]

# Verify sizes and class distribution
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("Train Cold Events:", train_df[train_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Train Non-Cold Events:", train_df[train_df['Cold_Event'] == 0]['Cold_Event'].count())
print("Val Cold Events:", val_df[val_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Val Non-Cold Events:", val_df[val_df['Cold_Event'] == 0]['Cold_Event'].count())
print("Test Cold Events:", test_df[test_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Test Non-Cold Events:", test_df[test_df['Cold_Event'] == 0]['Cold_Event'].count())

Train size: 44226
Validation size: 1830
Test size: 7795
Train Cold Events: 10290
Train Non-Cold Events: 33936
Val Cold Events: 387
Val Non-Cold Events: 1443
Test Cold Events: 1657
Test Non-Cold Events: 6138


In [15]:
import pandas as pd
import numpy as np
from prophet import Prophet
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

# Assume df_final is your processed dataframe from FeatureEngineering and ColdEventDetector
# Define split dates
train_end_date = pd.to_datetime('2020-01-01')
val_end_date = pd.to_datetime('2021-01-01')

# Split into train, validation, and test
train_df = df[df['DATE'] < train_end_date]
val_df = df[(df['DATE'] >= train_end_date) & (df['DATE'] < val_end_date)]
test_df = df[df['DATE'] >= val_end_date]

# Verify sizes and class distribution
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("Train Cold Events:", train_df[train_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Train Non-Cold Events:", train_df[train_df['Cold_Event'] == 0]['Cold_Event'].count())
print("Val Cold Events:", val_df[val_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Val Non-Cold Events:", val_df[val_df['Cold_Event'] == 0]['Cold_Event'].count())
print("Test Cold Events:", test_df[test_df['Cold_Event'] == 1]['Cold_Event'].count())
print("Test Non-Cold Events:", test_df[test_df['Cold_Event'] == 0]['Cold_Event'].count())

# Step 1: Time Series Forecasting with Prophet
def prophet_forecast(station_df, forecast_start, forecast_end):
    prophet_df = station_df[['DATE', 'TMIN']].rename(columns={'DATE': 'ds', 'TMIN': 'y'})
    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, changepoint_prior_scale=0.05)
    model.fit(prophet_df)
    future_dates = pd.date_range(start=forecast_start, end=forecast_end, freq='D')
    future = pd.DataFrame({'ds': future_dates})
    forecast = model.predict(future)
    return forecast[['ds', 'yhat']]

# Forecast for validation period (2020)
val_forecasts = train_df.groupby('Station_ID').apply(
    lambda x: prophet_forecast(x, '2020-01-01', '2020-12-31')
).reset_index()
df_val_future = val_forecasts.rename(columns={'ds': 'DATE', 'yhat': 'TMIN_pred'})
df_val_future['DATE'] = pd.to_datetime(df_val_future['DATE'])

# Forecast for test period (2021–2025)
test_forecasts = train_df.groupby('Station_ID').apply(
    lambda x: prophet_forecast(x, '2021-01-01', '2025-04-08')
).reset_index()
df_test_future = test_forecasts.rename(columns={'ds': 'DATE', 'yhat': 'TMIN_pred'})
df_test_future['DATE'] = pd.to_datetime(df_test_future['DATE'])

# Step 2: Generate Features for Validation and Test
def generate_future_features(df_future, historical_df):
    fe_future = FeatureEngineering(df_future.rename(columns={'TMIN_pred': 'TMIN'}))
    df_future_engineered = fe_future.apply_all_features()
    return df_future_engineered

# Assuming FeatureEngineering class is defined as in previous responses
df_val_future_engineered = generate_future_features(df_val_future, train_df)
df_test_future_engineered = generate_future_features(df_test_future, train_df)

# Step 3: Train XGBoost with Validation
X_train = train_df[['TMIN', 'Rolling_10thPercentile_TMIN_7', 'TMIN_Rolling_30_Diff', 'Seasonal_TMIN_Anomaly']]
y_train = train_df['Cold_Event']
X_val = val_df[['TMIN', 'Rolling_10thPercentile_TMIN_7', 'TMIN_Rolling_30_Diff', 'Seasonal_TMIN_Anomaly']]
y_val = val_df['Cold_Event']
X_test = test_df[['TMIN', 'Rolling_10thPercentile_TMIN_7', 'TMIN_Rolling_30_Diff', 'Seasonal_TMIN_Anomaly']]
y_test = test_df['Cold_Event']

# Class weight from full dataset
pos_weight = 41517 / 12334  # ~3.37
xgb = XGBClassifier(
    scale_pos_weight=pos_weight,
    objective='binary:logistic',
    max_depth=6,
    n_estimators=100,
    eval_metric='aucpr'
)

# Fit with early stopping using validation set
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=10, verbose=False)

# Calibrate probabilities using validation set
calibrated = CalibratedClassifierCV(xgb, method='isotonic', cv='prefit')
calibrated.fit(X_val, y_val)

# Step 4: Evaluate on Validation and Test
# Validation predictions
probs_val = calibrated.predict_proba(X_val)[:, 1] * 100
val_df['Cold_Event_Probability'] = probs_val
val_auc = roc_auc_score(y_val, probs_val / 100)  # Convert back to 0-1 for scoring
val_brier = brier_score_loss(y_val, probs_val / 100)

# Test predictions
probs_test = calibrated.predict_proba(X_test)[:, 1] * 100
test_df['Cold_Event_Probability'] = probs_test
test_auc = roc_auc_score(y_test, probs_test / 100)
test_brier = brier_score_loss(y_test, probs_test / 100)

# Future predictions (2021–2025 from Prophet)
X_future = df_test_future_engineered[['TMIN', 'Rolling_10thPercentile_TMIN_7', 'TMIN_Rolling_30_Diff', 'Seasonal_TMIN_Anomaly']]
probs_future = calibrated.predict_proba(X_future)[:, 1] * 100
df_test_future_engineered['Cold_Event_Probability'] = probs_future

# Results
print("\nValidation Performance (2020):")
print(f"AUC-ROC: {val_auc:.3f}, Brier Score: {val_brier:.3f}")
print(val_df[['DATE', 'Station_ID', 'Cold_Event_Probability']].head())

print("\nTest Performance (2021–2025):")
print(f"AUC-ROC: {test_auc:.3f}, Brier Score: {test_brier:.3f}")
print(test_df[['DATE', 'Station_ID', 'Cold_Event_Probability']].head())

print("\nFuture Predictions (Last 7 days of 2025):")
print(df_test_future_engineered[['DATE', 'Station_ID', 'Cold_Event_Probability']].tail(7))

Train size: 44226
Validation size: 1830
Test size: 7795
Train Cold Events: 10290
Train Non-Cold Events: 33936
Val Cold Events: 387
Val Non-Cold Events: 1443
Test Cold Events: 1657
Test Non-Cold Events: 6138


23:48:37 - cmdstanpy - INFO - Chain [1] start processing
23:48:37 - cmdstanpy - INFO - Chain [1] done processing
23:48:38 - cmdstanpy - INFO - Chain [1] start processing
23:48:38 - cmdstanpy - INFO - Chain [1] done processing
23:48:38 - cmdstanpy - INFO - Chain [1] start processing
23:48:38 - cmdstanpy - INFO - Chain [1] done processing
23:48:39 - cmdstanpy - INFO - Chain [1] start processing
23:48:39 - cmdstanpy - INFO - Chain [1] done processing
23:48:40 - cmdstanpy - INFO - Chain [1] start processing
23:48:42 - cmdstanpy - INFO - Chain [1] done processing
/tmp/ipykernel_48546/2344744467.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_forecasts = train_df.groupby('Station_ID').apply(
2

KeyError: 'LATITUDE'